Ce notebook permet de lancer un LDA uniquement sur les titres du sommaire et nn plus tout le contenu. 
Le but est d'obtenir des résultats mons bruités avec un sommaire qui porte déjà l'info nécessaire. 

In [1]:
!uv pip install -q nltk gensim pyLDAvis unidecode matplotlib seaborn pandas pyarrow

In [2]:
from gensim.models import CoherenceModel, LdaModel, LdaMulticore
from gensim.utils import simple_preprocess
from pathlib import Path
import gensim
import gensim.corpora as corpora
import json
import matplotlib.pyplot  as plt
import numpy as np
import os
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim
import pyLDAvis.gensim_models as gensimvis
import warnings

In [3]:
INTERMEDIATE_DATA_DIR="intermediate_data"

# Utils LDA

In [4]:
def lda_model(processed_texts, num_topics=5, passes=10):
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    warnings.filterwarnings('ignore')
    dictionary = corpora.Dictionary(processed_texts)
    corpus = [dictionary.doc2bow(text) for text in processed_texts]
    model = LdaMulticore(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=passes ,workers=10, eta='auto' ,chunksize=1000)
    for topic in model.print_topics(num_words=5):
        print(topic)
    return model, corpus, dictionary
    
def visualize_lda(model, corpus, dictionary):
    pyLDAvis.enable_notebook()
    vis_data = gensimvis.prepare(model, corpus, dictionary)
    return pyLDAvis.display(vis_data)

In [5]:
def compute_coherence_values(dictionary, corpus, texts, max_topics=10):
    coherence_scores = []
    for num_topics in range(2, max_topics + 1):
        lda_model = LdaMulticore(corpus=corpus, id2word=dictionary, num_topics=num_topics, random_state=42, passes=5 ,workers=10, eta='auto' ,chunksize=1000)
        coherence_model = CoherenceModel(model=lda_model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_score = coherence_model.get_coherence()
        coherence_scores.append((num_topics, coherence_score))
        print(f"Num Topics: {num_topics}, Coherence Score: {coherence_score:.4f}")
    
    return coherence_scores

# Pour HS

In [6]:
df_hs = pd.read_parquet(f"{INTERMEDIATE_DATA_DIR}/processed_titles_hs.parquet")


In [7]:
all_chunks_hs = [list(title) for summary in df_hs["lda_documents"] for title in summary]


In [8]:
model_hs, corpus_hs, dictionary_hs = lda_model(all_chunks_hs)

(0, '0.134*"temp" + 0.129*"travail" + 0.050*"jours" + 0.037*"organisation" + 0.020*"partiel"')
(1, '0.041*"congés" + 0.041*"application" + 0.034*"champ" + 0.030*"droit" + 0.030*"rémunération"')
(2, '0.085*"travail" + 0.082*"durée" + 0.081*"accord" + 0.030*"entreprise" + 0.022*"hebdomadaire"')
(3, '0.043*"repos" + 0.043*"période" + 0.039*"jours" + 0.038*"absence" + 0.032*"cours"')
(4, '0.162*"heures" + 0.101*"supplémentaires" + 0.044*"contingent" + 0.029*"salariés" + 0.027*"annuel"')


In [9]:
visualize_lda(model_hs, corpus_hs, dictionary_hs)

In [10]:
compute_coherence_values(dictionary_hs, corpus_hs, all_chunks_hs, max_topics=20)

Num Topics: 2, Coherence Score: 0.3288
Num Topics: 3, Coherence Score: 0.3545
Num Topics: 4, Coherence Score: 0.3877
Num Topics: 5, Coherence Score: 0.3289
Num Topics: 6, Coherence Score: 0.3678
Num Topics: 7, Coherence Score: 0.3405
Num Topics: 8, Coherence Score: 0.3638
Num Topics: 9, Coherence Score: 0.3801
Num Topics: 10, Coherence Score: 0.3809
Num Topics: 11, Coherence Score: 0.3823
Num Topics: 12, Coherence Score: 0.3863
Num Topics: 13, Coherence Score: 0.3962
Num Topics: 14, Coherence Score: 0.3901
Num Topics: 15, Coherence Score: 0.4036
Num Topics: 16, Coherence Score: 0.3998
Num Topics: 17, Coherence Score: 0.4108
Num Topics: 18, Coherence Score: 0.3991
Num Topics: 19, Coherence Score: 0.4174
Num Topics: 20, Coherence Score: 0.4116


[(2, 0.32881237135992014),
 (3, 0.35451537698541197),
 (4, 0.3876698148912711),
 (5, 0.3288606574390373),
 (6, 0.3678201841165211),
 (7, 0.34045755551310236),
 (8, 0.36376142971860437),
 (9, 0.3800771107095664),
 (10, 0.3808740590531333),
 (11, 0.3822804069956574),
 (12, 0.3862838333147775),
 (13, 0.3962221618464709),
 (14, 0.39010156054306994),
 (15, 0.40359200135174106),
 (16, 0.39984113880733085),
 (17, 0.41084268740906404),
 (18, 0.3990884575943265),
 (19, 0.41735297998473153),
 (20, 0.4115997551542299)]

In [13]:
model_hs, corpus_hs, dictionary_hs = lda_model(all_chunks_hs, num_topics= 19)

(0, '0.178*"jours" + 0.055*"nombre" + 0.053*"travaillés" + 0.049*"repos" + 0.039*"rtt"')
(1, '0.163*"rémunération" + 0.130*"congés" + 0.073*"payés" + 0.054*"prime" + 0.047*"lissage"')
(2, '0.081*"repos" + 0.050*"hebdomadaire" + 0.039*"quotidien" + 0.025*"professionnelle" + 0.020*"indemnité"')
(3, '0.151*"jours" + 0.113*"forfait" + 0.099*"repos" + 0.050*"convention" + 0.041*"annuel"')
(4, '0.139*"période" + 0.100*"absence" + 0.096*"cours" + 0.088*"référence" + 0.028*"sortie"')
(5, '0.182*"application" + 0.148*"champ" + 0.033*"salarié" + 0.032*"temp" + 0.029*"plein"')
(6, '0.101*"dépôt" + 0.087*"accord" + 0.081*"publicité" + 0.057*"année" + 0.048*"repos"')
(7, '0.317*"heures" + 0.188*"supplémentaires" + 0.057*"complémentaires" + 0.028*"supplementaires" + 0.021*"décompte"')
(8, '0.117*"disposition" + 0.050*"relative" + 0.045*"télétravail" + 0.037*"finale" + 0.034*"salary"')
(9, '0.150*"temp" + 0.107*"partiel" + 0.079*"salariés" + 0.057*"droit" + 0.043*"déconnexion"')
(10, '0.219*"accord" 

In [19]:
visualize_lda(model_hs, corpus_hs, dictionary_hs)

Exception ignored in: <function ResourceTracker.__del__ at 0x7f97a66236a0>
Traceback (most recent call last):
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7feeb70676a0>
Traceback (most recent call last):
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/python/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7f0520f036a0>
Traceback (most recent call last):
  File

In [18]:
#Pour auvegarder dans un fichier HTML

import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
vis_data = gensimvis.prepare(model_hs, corpus_hs, dictionary_hs)
pyLDAvis.save_html(vis_data, 'lda_visualization.html')